# 🚀 Servidor de IA na Nuvem (Google Colab GPU)
### Tradução de Mangás e Manhwas Sem Censura (Qwen 2.5 + Dolphin 2.9)

Este notebook executa os modelos **Qwen 2.5** (especialista em Japonês e Coreano) e **Dolphin 2.9** diretamente na GPU gratuita do Google Colab e gera uma URL pública para você conectar ao seu tradutor sem gastar recursos do seu computador!

---
### 📋 Passo a Passo:
1. No menu superior do Colab, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** e selecione **T4 GPU**.
2. Clique no botão de execução **Play [ ▶ ]** na célula abaixo.
3. Aguarde cerca de 2 minutos até que a mensagem exiba a sua **URL Pública** (exemplo: `https://xxxx.trycloudflare.com`).
4. Abra o arquivo `config.json` no seu computador e cole essa URL no campo `ollama_host`:
   ```json
   "ollama_host": "https://xxxx.trycloudflare.com"
   ```
5. Pronto! Execute o seu `Traduzir_Manga.bat`. Toda a tradução rodará na GPU da nuvem do Google!

In [ ]:
# @title ▶ Iniciar Servidor de IA (Qwen 2.5 / Dolphin) + Túnel Nuvem (Clique para Rodar)
import subprocess
import time
import os
import re

print("[*] [1/4] Instalando Ollama no Google Colab...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("[*] [2/4] Iniciando servico Ollama em segundo plano...")
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(4)

print("[*] [3/4] Baixando modelos de IA recomendados (Qwen 2.5 e Dolphin 2.9)...")
subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
subprocess.run(["ollama", "pull", "dolphin-llama3:8b"], check=True)

print("[*] [4/4] Configurando tunel seguro Cloudflare...")
subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:11434"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("\n" + "="*65)
print(" AGUARDANDO URL PUBLICA DO TUNEL...")
print("="*65)

found = False
for line in tunnel.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*65)
        print("  🎉 SUCESSO! SEU SERVIDOR DE IA ESTA ATIVO NA NUVEM!")
        print(f"  URL: {public_url}")
        print("="*65)
        print("\nComo conectar no seu computador:")
        print("1. Abra o arquivo 'config.json' na pasta do tradutor.")
        print(f'2. Altere "ollama_host" para: "{public_url}"')
        print("3. Execute o 'Traduzir_Manga.bat' normalmente.")
        print("\n(Deixe esta aba do Colab aberta enquanto estiver traduzindo!)")
        found = True
        break

if not found:
    print("[!] Nao foi possivel obter a URL do tunel automaticamente. Verifique os logs.")
